# Rotate API keys → push to the VM

**One-shot notebook for rotating Bybit API keys on the trading VM.**

What this does:
1. Mounts your Google Drive and reads your VM SSH private key from `My Drive/ICT_Bot_Secrets/`.
2. Reads your API keys + connection details from Colab Secrets.
3. Generates a fresh `.env` (the file the systemd unit reads) and `.env.live` (legacy convention).
4. Uploads both to the VM via SSH (atomic write, owner-only permissions).
5. Restarts the trader systemd unit and verifies it's active.

**How to run:** `Runtime → Run all`. The first cell triggers a one-click "Allow Drive access" dialog the first time you run it in a session — click *Allow*. After that, no further interaction.

**If your SSH key isn't in Drive** (or Drive can't be mounted), the notebook automatically falls back to a file picker that lets you upload the key from your computer.

---

**Required Colab Secrets** (set them once via `Tools → Secrets`, toggle *Notebook access* on):

| Name | What it holds |
|---|---|
| `BYBIT_API_KEY_1`     | Bybit API key for account `bybit_1` (turtle_soup) |
| `BYBIT_API_SECRET_1`  | Bybit API secret for account `bybit_1` |
| `BYBIT_API_KEY_2`     | Bybit API key for account `bybit_2` (vwap) |
| `BYBIT_API_SECRET_2`  | Bybit API secret for account `bybit_2` |
| `TELEGRAM_BOT_TOKEN`  | Bot token from @BotFather |
| `TELEGRAM_CHAT_ID`    | Your Telegram chat id |
| `VM_SSH_HOST`         | The VM's hostname or public IP |
| `VM_SSH_USER`         | SSH user on the VM (usually `ubuntu`) |

**Required SSH key file** — preferred location:

`My Drive/ICT_Bot_Secrets/ict-bot-ovm-private.key`

(also accepted in the same folder: `vm_ssh_key`, `id_rsa`, `id_ed25519`, `id_ecdsa` — first match wins). If your file is named something else, set the optional `SSH_KEY_FILE` Colab Secret to its filename. If the notebook can't find it in Drive, it'll pop a file-picker so you can upload from your computer.

**Optional Colab Secrets** (skip if unused):

| Name | What it holds |
|---|---|
| `SSH_KEY_FILE`           | A custom SSH key filename if yours doesn't match the defaults above. |
| `BREAKOUT_API_KEY_1`     | Prop-firm API key (currently disabled in `accounts.yaml`) |
| `BREAKOUT_API_SECRET_1`  | Prop-firm API secret |
| `NEWS_API_KEY`           | NewsAPI key (only if you've enabled the news layer) |

---

**Security:** Colab Secrets are scoped to your Google account and never appear in the notebook source. The SSH key is read from Drive (or your local upload), copied to a Colab tempdir with `0600` perms only for the duration of the SSH call, then wiped in a `finally` block. This notebook never prints, logs, or commits any secret value.



In [ ]:
# Step 1A: Mount Google Drive FIRST so the auth dialog pops before
# anything else runs. This is intentionally a separate, tiny cell so
# the dialog cannot be missed.
import os

from google.colab import drive

print('⏳ Mounting Google Drive (a one-click "Allow" dialog will pop the first time per session)…')
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f'⚠️  drive.mount() raised: {exc}')

# Verify the mount actually succeeded. drive.mount() is supposed to
# block until the auth flow finishes, but in some Colab sessions a
# stale token can let it return early without a fresh consent.
if not os.path.exists('/content/drive/MyDrive'):
    print('Drive not visible after first mount; forcing a re-auth…')
    try:
        drive.mount('/content/drive', force_remount=True)
    except Exception as exc:
        print(f'⚠️  force-remount also failed: {exc}')

drive_mounted = os.path.exists('/content/drive/MyDrive')
if drive_mounted:
    print('✅ Drive mounted.')
else:
    print('⚠️  Drive is NOT mounted. The next cell will prompt you to '
          'upload the SSH key from your computer instead.')

In [ ]:
# Step 1B: Locate the SSH key. Drive first; if Drive lookup fails
# (file not present, or Drive not mounted), pop a file-picker so the
# operator can upload it from their computer.
import os
from google.colab import userdata

# ict-bot-ovm-private.key is the first candidate because that's the
# canonical OCI key filename. The other names cover the standard SSH
# defaults so any common key works without renaming.
DEFAULT_SSH_KEY_NAMES = [
    'ict-bot-ovm-private.key',
    'vm_ssh_key', 'id_rsa', 'id_ed25519', 'id_ecdsa',
]
DRIVE_SECRETS_FOLDER = '/content/drive/MyDrive/ICT_Bot_Secrets'
FALLBACK_FOLDER = '/content'

# Optional filename override.
override_name = None
try:
    override_name = userdata.get('SSH_KEY_FILE')
except Exception:
    pass

candidate_names = []
if override_name:
    candidate_names.append(override_name)
candidate_names.extend(DEFAULT_SSH_KEY_NAMES)

# A. Try Drive first.
ssh_key_path = None
key_source = None
if drive_mounted:
    for name in candidate_names:
        path = os.path.join(DRIVE_SECRETS_FOLDER, name)
        if os.path.exists(path):
            ssh_key_path = path
            key_source = 'Drive'
            break

# B. Try /content/ (in case the operator already drag-dropped one).
if ssh_key_path is None:
    for name in candidate_names:
        path = os.path.join(FALLBACK_FOLDER, name)
        if os.path.exists(path):
            ssh_key_path = path
            key_source = 'Files panel'
            break

# C. Last resort: pop a file picker. This is interactive — the cell
# blocks until the operator clicks "Choose Files" and picks one.
if ssh_key_path is None:
    print('SSH key not found in Drive or /content/.')
    print('Looked for these names in '
          + (f'{DRIVE_SECRETS_FOLDER}/ and ' if drive_mounted else '')
          + f'{FALLBACK_FOLDER}/:')
    for name in candidate_names:
        print(f'  - {name}')
    print()
    print('Falling back to direct upload. Click "Choose Files" below '
          'and pick your VM SSH private key file from your computer.')
    print()
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit(
            'No file uploaded. Run this cell again and pick the key file, '
            'or place it in My Drive/ICT_Bot_Secrets/ as '
            'ict-bot-ovm-private.key and re-run all cells.'
        )
    uploaded_name = next(iter(uploaded.keys()))
    ssh_key_path = os.path.join(FALLBACK_FOLDER, uploaded_name)
    key_source = 'computer upload'

# D. Sanity check: must look like a private key, not a public key.
with open(ssh_key_path, 'r') as f:
    first_line = f.readline().strip()
if not first_line.startswith('-----BEGIN'):
    raise SystemExit(
        f'{ssh_key_path} does not look like a private key (first line: '
        f'\"{first_line[:40]}\"). Expected something like \"-----BEGIN '
        f'OPENSSH PRIVATE KEY-----\". Did you point at your public key '
        f'(.pub) instead of the private one?'
    )
if 'PUBLIC KEY' in first_line:
    raise SystemExit(
        f'{ssh_key_path} is a PUBLIC key. Use the matching PRIVATE key '
        f'(no .pub extension).'
    )

print(f'✅ SSH key located: {ssh_key_path}')
print(f'   Source: {key_source}')
print(f'   Looks like a valid private key.')

In [ ]:
# Step 1C: Load Colab Secrets and validate.
from google.colab import userdata

REQUIRED = [
    'BYBIT_API_KEY_1', 'BYBIT_API_SECRET_1',
    'BYBIT_API_KEY_2', 'BYBIT_API_SECRET_2',
    'TELEGRAM_BOT_TOKEN', 'TELEGRAM_CHAT_ID',
    'VM_SSH_HOST', 'VM_SSH_USER',
]
OPTIONAL = [
    'BREAKOUT_API_KEY_1', 'BREAKOUT_API_SECRET_1',
    'NEWS_API_KEY',
]

secrets = {}
missing = []
for name in REQUIRED:
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    if not v:
        missing.append(name)
    else:
        secrets[name] = v

for name in OPTIONAL:
    try:
        v = userdata.get(name)
        if v:
            secrets[name] = v
    except Exception:
        pass

if missing:
    raise SystemExit(
        'Missing required Colab Secrets:\n  - '
        + '\n  - '.join(missing)
        + '\n\nFix: Tools → Secrets, add each missing secret with exactly that name '
        + '(toggle Notebook access on), then run all again.'
    )

loaded_required = [n for n in REQUIRED if n in secrets]
loaded_optional = [n for n in OPTIONAL if n in secrets]
print(f'Loaded {len(loaded_required)} required + {len(loaded_optional)} optional secrets from Colab.')
print(f'  Required:  {sorted(loaded_required)}')
print(f'  Optional:  {sorted(loaded_optional) or "(none)"}')
print()
print(f'VM target: {secrets["VM_SSH_USER"]}@{secrets["VM_SSH_HOST"]}')

In [ ]:
# Step 2: Build the .env.live content.
#
# Matches the keys produced by scripts/render_env_from_master.py for the
# vwap_btcusd_live profile. Risk caps are hardcoded here to the production
# defaults from config/master-secrets.template.yaml::risk.vwap_btcusd. Edit
# this cell if you intentionally want to change a non-secret default.

PRODUCTION_DEFAULTS = {
    'ENVIRONMENT': 'production',
    'EXCHANGE': 'bybit',
    'MODE': 'LIVE',
    'DRY_RUN': 'false',
    'ALLOW_LIVE_TRADING': 'true',
    'BYBIT_TESTNET': 'false',
    'STRATEGY': 'vwap',
    'SYMBOL': 'BTCUSDT',
    'TIMEFRAME': '1m',
    'DATA_DIR': 'data/',
    'MODEL_DIR': 'ml/models/',
    'LOG_DIR': 'logs/',
    'DB_PATH': 'data/trading.db',
    'MAX_POSITION_USD': '50',
    'MAX_DAILY_LOSS_USD': '25',
    'RISK_PER_TRADE': '0.005',
    'MAX_QTY': '0.001',
    'MAX_OPEN_POSITIONS': '1',
    'NEWS_ENABLED': 'false',
    'NEWS_API_KEY': '',
    # Legacy single-account vars (back-compat with code paths that still
    # read the unsuffixed names; mirrors render_env_from_master.py).
    'BYBIT_API_KEY': secrets['BYBIT_API_KEY_1'],
    'BYBIT_API_SECRET': secrets['BYBIT_API_SECRET_1'],
    'BYBIT_BASE_URL': 'https://api.bybit.com',
}

# Per-account credentials (the names accounts.yaml looks up via api_key_env).
PER_ACCOUNT_KEYS = [
    'BYBIT_API_KEY_1', 'BYBIT_API_SECRET_1',
    'BYBIT_API_KEY_2', 'BYBIT_API_SECRET_2',
    'BREAKOUT_API_KEY_1', 'BREAKOUT_API_SECRET_1',
]

TELEGRAM_KEYS = ['TELEGRAM_BOT_TOKEN', 'TELEGRAM_CHAT_ID']


def quote_if_needed(val: str) -> str:
    if any(c in val for c in ' \t#$`\'"\\'):
        return '"' + val.replace('\\', '\\\\').replace('"', '\\"') + '"'
    return val


lines = []
for k, v in PRODUCTION_DEFAULTS.items():
    if k == 'NEWS_API_KEY' and 'NEWS_API_KEY' in secrets:
        v = secrets['NEWS_API_KEY']
    if k == 'NEWS_ENABLED' and 'NEWS_API_KEY' in secrets:
        v = 'true'
    lines.append(f'{k}={quote_if_needed(str(v))}')
for k in TELEGRAM_KEYS:
    lines.append(f'{k}={quote_if_needed(secrets[k])}')
for k in PER_ACCOUNT_KEYS:
    if k in secrets:
        lines.append(f'{k}={quote_if_needed(secrets[k])}')

env_content = '\n'.join(lines) + '\n'

# Print only the KEY NAMES, never values. Operator can confirm shape
# without leaking secrets.
key_names = [line.split('=', 1)[0] for line in lines]
print(f'Generated .env.live with {len(key_names)} variables:')
for name in key_names:
    print(f'  {name}')

In [ ]:
# Step 3: Push the .env file to the VM via SCP, restart the trader, verify.
#
# IMPORTANT: ict-trader-live.service systemd unit reads
# EnvironmentFile=/home/<user>/ict-trading-bot/.env (NOT .env.live).
# We write to BOTH .env (the file systemd actually loads) and .env.live
# (legacy convention used by render_env_from_master.py + pipeline.py
# load_dotenv) so every code path sees the new credentials.
import os
import shutil
import stat
import subprocess
import tempfile

REPO_PATH_ON_VM = '/home/' + secrets['VM_SSH_USER'] + '/ict-trading-bot'
REMOTE_TARGETS = [
    REPO_PATH_ON_VM + '/.env',         # systemd EnvironmentFile reads this
    REPO_PATH_ON_VM + '/.env.live',    # back-compat for pipeline.py + render script
]
SERVICE_NAME = 'ict-trader-live.service'

# Stage SSH key + .env in a Colab tempdir with 0600 perms — ssh refuses
# key files with broader permissions, and Drive-mounted files don't
# honour chmod (they always look 0644).
tmpdir = tempfile.mkdtemp(prefix='rotate-keys-')
key_path = os.path.join(tmpdir, 'vm_key')
env_path = os.path.join(tmpdir, '.env')

shutil.copy(ssh_key_path, key_path)
os.chmod(key_path, stat.S_IRUSR | stat.S_IWUSR)  # 0600

with open(env_path, 'w') as f:
    f.write(env_content)
os.chmod(env_path, stat.S_IRUSR | stat.S_IWUSR)

host = secrets['VM_SSH_HOST']
user = secrets['VM_SSH_USER']
ssh_target = f'{user}@{host}'
ssh_opts = [
    '-i', key_path,
    '-o', 'StrictHostKeyChecking=accept-new',
    '-o', 'UserKnownHostsFile=' + os.path.join(tmpdir, 'known_hosts'),
    '-o', 'ConnectTimeout=15',
    '-o', 'BatchMode=yes',
]


def _redact(text: str) -> str:
    """Strip any secret values that may have leaked into stderr."""
    if not text:
        return text
    for v in secrets.values():
        if v and len(str(v)) > 8:
            text = text.replace(str(v), '<REDACTED>')
    return text


def run_ssh(cmd: str, *, label: str):
    full = ['ssh'] + ssh_opts + [ssh_target, cmd]
    proc = subprocess.run(full, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f'❌ {label} failed (exit {proc.returncode})')
        print(_redact(proc.stderr or proc.stdout)[:500])
        raise SystemExit(f'{label} failed')
    return (proc.stdout or '').strip()


def run_scp(local: str, remote: str, *, label: str):
    full = ['scp'] + ssh_opts + [local, f'{ssh_target}:{remote}']
    proc = subprocess.run(full, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f'❌ {label} failed (exit {proc.returncode})')
        print(_redact(proc.stderr or proc.stdout)[:500])
        raise SystemExit(f'{label} failed')


try:
    print(f'⏳ Connecting to {ssh_target} …')
    run_ssh('echo connected', label='SSH connectivity')
    print('✅ SSH OK')

    for remote_final in REMOTE_TARGETS:
        remote_tmp = remote_final + '.tmp'
        print(f'⏳ Uploading {os.path.basename(remote_final)} (atomic) …')
        run_scp(env_path, remote_tmp, label=f'SCP upload {remote_final}')
        run_ssh(f'chmod 600 {remote_tmp} && mv {remote_tmp} {remote_final}',
                label=f'atomic rename {remote_final}')
        print(f'✅ Wrote {remote_final}')

    print(f'⏳ Restarting {SERVICE_NAME} …')
    run_ssh(f'sudo -n systemctl restart {SERVICE_NAME}',
            label='service restart')
    print(f'✅ Restart issued')

    print(f'⏳ Verifying service is active …')
    state = run_ssh(f'sudo -n systemctl is-active {SERVICE_NAME}',
                    label='is-active check')
    print(f'✅ Service state: {state}')
    if state != 'active':
        print('⚠️  Service is not "active" — check journalctl on the VM.')
finally:
    # Wipe the tempdir copy of the key + env. The original in Drive (or
    # /content/) is untouched, so re-running the notebook just works
    # without re-uploading.
    shutil.rmtree(tmpdir, ignore_errors=True)

print('\nDone. Open Telegram and run /accounts_status to verify each account shows a real balance.')


## Verification

After the cells above complete, in Telegram:

1. **`/accounts_status`** — every account should show ✅ with a real USDT balance.
2. **`/smoke_test`** — each account should return ✅ `rejected_too_small` (Bybit accepted the request and rejected for size, which proves the keys work).

If any account still shows ❌, the new diagnostic message names the exact reason (missing env var, `Bybit error retCode=10003`, network, etc.). See the troubleshooting section in [`docs/operator/colab-key-rotation.md`](https://github.com/the-lizardking/ict-trading-bot/blob/main/docs/operator/colab-key-rotation.md).

## When to re-run

- You rotated a Bybit API key (update the matching Colab Secret first).
- You changed the VM's SSH key (replace the file in Drive with the new one — same filename — first).
- You added a new account to `config/accounts.yaml`.
- The bot is alerting on `bybit_get_wallet_balance_failed` with `retCode=10003` (key invalid).

## Cleaning up

Nothing to clean. The SSH key in Drive stays where it is (you put it there intentionally). The Colab tempdir copy is wiped in the `finally` block of step 3.